# BioMapper vs Monti et al. 2026: cross-cohort overlap benchmark

This notebook reproduces the BioMapper side of the cross-cohort comparison against
Monti et al. 2026 (*Metabolomic signatures of extreme old age: findings from the New England
Centenarian Study*, GeroScience, doi:10.1007/s11357-026-02174-2). NECS is the reference cohort and
four cohorts are linked to it: Arivale, Xu et al. 2022, LLFS and BLSA.

It reports **two layers, and they answer different questions.** Keep them apart when quoting.

| Layer | Question | Cohorts | What the number is |
|---|---|---|---|
| **(a) Overlap** | How many NECS metabolites does each method link to the other cohort? | all four | **Linked-count coverage**, not correctness. A link is two names resolving to a shared identifier. |
| **(b) Certification** | Of the links whose structure can be checked outside the graph, how many hold up? | Arivale only | A **correctness check on the adjudicable subset**, not accuracy over all links, and not fully knowledge-graph independent (see Part 5). |

Xu, LLFS and BLSA publish names only, with no vendor identifiers. Their links can be counted but
never structurally certified, so they are **refused by construction** in layer (b). That is a
property of the source panels, not a failed certification.

### How this notebook runs by default

A full re-resolution of all five panels cost 5.6 hours of API time. The default is therefore:

1. **Replay** of the full pinned results shipped in `notebooks/data/monti_overlap/`.
2. A **small live subset** (17 NECS and 50 Arivale names, a few minutes) resolved against the
   keyless deployment and checked against the pinned run.
3. Recomputations that need no API at all (Monti's published comparator, Arm B for Arivale).

The full run is available behind `RUN_FULL`, and is off by default.

---

## Part 0: Install and select the kernel

Run from the repository root.

```bash
uv venv .venv --python 3.11
uv pip install --python .venv/bin/python \
    'biomapper[benchmarks,notebook] @ git+https://github.com/trentleslie/biomapper@3761dc87' \
    ipykernel nbconvert
# Once biomapper 1.5.4 is on PyPI, the pin becomes 'biomapper[benchmarks,notebook]==1.5.4'.

# Register that environment under the kernel name this notebook DECLARES.
.venv/bin/python -m ipykernel install --user \
    --name biomapper-monti --display-name "Python 3.11 (biomapper, Monti benchmark)"

# Execute headless:
.venv/bin/jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=1800 notebooks/monti_overlap_benchmark.ipynb
```

The registration step is required. Jupyter and nbconvert do not run the interpreter that launched
them; they start the kernel named in the notebook's `metadata.kernelspec`, here `biomapper-monti`.
Without it the notebook can run on some other Python, unpinned, while still printing pinned
provenance. The first code cell refuses to continue in that case.

In [1]:
import hashlib
import json
import os
import sys
import time
import urllib.request
from importlib import metadata
from pathlib import Path

import nest_asyncio
import pandas as pd

import biomapper

# Several library helpers call asyncio.run(), which Jupyter's own running loop would reject.
nest_asyncio.apply()

# Accept either the published release or the exact commit this notebook was built against.
ACCEPTED_VERSION = "1.5.4"
ACCEPTED_COMMIT = "3761dc87621258d00d4e021fab4260ae54471052"

dist = metadata.distribution("biomapper")
direct_url = json.loads(dist.read_text("direct_url.json") or "{}")
installed_commit = direct_url.get("vcs_info", {}).get("commit_id")

print(f"python             {sys.executable}")
print(f"biomapper          {biomapper.__version__}")
print(f"installed from     {direct_url.get('url', 'index (PyPI)')}")
print(f"installed commit   {installed_commit}")

if sys.prefix == sys.base_prefix:
    raise RuntimeError(
        "This kernel is a system Python, not a virtual environment. Select the 'biomapper-monti' "
        "kernel (Part 0) so the run is pinned."
    )
if not (biomapper.__version__ == ACCEPTED_VERSION or installed_commit == ACCEPTED_COMMIT):
    raise RuntimeError(
        f"biomapper {biomapper.__version__} @ {installed_commit} is not the pinned build "
        f"({ACCEPTED_VERSION} or commit {ACCEPTED_COMMIT[:8]}). Reinstall per Part 0."
    )
print("\npinned environment: OK")

python             /home/trentleslie/projects/biomapper-monti/.venv/bin/python
biomapper          1.5.3
installed from     https://github.com/trentleslie/biomapper
installed commit   3761dc87621258d00d4e021fab4260ae54471052

pinned environment: OK


In [2]:
# ---- Run configuration -------------------------------------------------------------------------
RUN_SUBSET = True   # live, small, a few minutes of API time
RUN_FULL = False    # live, all five panels, about 5.6 hours of API time. Needs the 4-cohort file.

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHIPPED = REPO_ROOT / "notebooks" / "data" / "monti_overlap"
INPUT_DIR = Path(os.environ.get("MONTI_INPUT_DIR", Path.home() / "monti_inputs"))
# The 4-cohort names spreadsheet (NECS / Xu / LLFS / BLSA columns). Not public; see Part 2.
COHORT_SPREADSHEET = os.environ.get("MONTI_COHORT_SPREADSHEET")

API_ENDPOINT = "https://biomapper.expertintheloop.io/api/v1"   # keyless
KESTREL_URL = "https://kestrel.krakenkg.com/api"                # keyless; never send it a key

# Every live run writes here by default, so its output survives the kernel.
RUN_STAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
OUT_DIR = Path(os.environ.get("MONTI_OUT_DIR", Path.home() / f"monti_overlap_runs/{RUN_STAMP}"))

print(f"shipped artifacts  {SHIPPED}")
print(f"public inputs      {INPUT_DIR}")
print(f"live output dir    {OUT_DIR}")

shipped artifacts  /home/trentleslie/projects/biomapper-monti/notebooks/data/monti_overlap
public inputs      /home/trentleslie/monti_inputs
live output dir    /home/trentleslie/monti_overlap_runs/20260928T200138Z


---

## Part 1: Pin the build before reading any output

Every number below is attributable to one KRAKEN build. The pinned run was answered by the build
recorded here, read from `GET {kestrel}/health`, never hardcoded into results.

The replayed numbers stay valid whatever the live service is serving today, because they carry
their own provenance. The **live** cells are only comparable to the pinned run when the live
build matches. When it does not, the live cells still run, and the mismatch is printed rather
than hidden.

In [3]:
from biomapper.benchmarks.provenance import fetch_kg_build_info

PINNED_BUILD = {
    "kestrel_version": "0.3.0",
    "kg_version": "2.3.0",
    "kraken_package_version": "2.2.0",
    "biolink_version": "4.2.5",
    "git_commit": "3dd08a5b",
}

kestrel_version, kg, health_error = fetch_kg_build_info(KESTREL_URL)
live_build = {
    "kestrel_version": kestrel_version,
    "kg_version": kg.kg_version,
    "kraken_package_version": kg.kraken_package_version,
    "biolink_version": kg.biolink_version,
    "git_commit": (kg.git_commit or "unknown")[:8],
}
if health_error:
    print(f"/health error: {health_error}")

rows = [
    {"field": k, "pinned": v, "live": live_build[k], "match": live_build[k] == v}
    for k, v in PINNED_BUILD.items()
]
print(pd.DataFrame(rows).to_string(index=False))
LIVE_BUILD_MATCHES = all(r["match"] for r in rows) and not health_error
print(f"\nlive build matches the pinned run: {LIVE_BUILD_MATCHES}")
if not LIVE_BUILD_MATCHES:
    print("Live cells below are NOT comparable to the pinned run; the replay is unaffected.")

                 field   pinned     live  match
       kestrel_version    0.3.0    0.3.0   True
            kg_version    2.3.0    2.3.0   True
kraken_package_version    2.2.0    2.2.0   True
       biolink_version    4.2.5    4.2.5   True
            git_commit 3dd08a5b 3dd08a5b   True

live build matches the pinned run: True


---

## Part 2: Inputs, where to get them, and their checksums

| Input | Used for | Source | Redistributable |
|---|---|---|---|
| Monti et al. 2026, **MOESM5** (Supplementary Table 5) | NECS names (`CHEMICAL_NAME`) and the NECS curated InChIKey gold | Springer ESM for doi:10.1007/s11357-026-02174-2 | Publisher supplement; downloaded, not shipped |
| Monti et al. 2026, **MOESM6** (Tables 1-2) | Monti's published overlap (`Table 2. Datasets`, `# Overlap`, NECS reference row) | same | Downloaded, not shipped |
| Watanabe et al. 2023, **Supplementary Data 2** | Arivale panel (`Arivale_Metabolomics`, `BiochemicalName`) and its vendor ids | Springer ESM for doi:10.1038/s41591-023-02248-0, CC BY | Yes (CC BY); downloaded here |
| **4-cohort names spreadsheet** | Xu, LLFS and BLSA names (and a NECS column identical to MOESM5) | Received by the BioMapper team on 2026-08-20; not publicly posted | **Not confirmed. Not shipped.** |

Two consequences:

- **NECS vs Arivale is fully reproducible from public inputs**, for both layers. The NECS column
  of the spreadsheet is identical to MOESM5 `CHEMICAL_NAME` (1495 of 1495 names), which the next
  cell does not assume but a full run re-checks.
- **Xu, LLFS and BLSA** overlap numbers are replayed from the pinned run. A reader without the
  spreadsheet can verify them only through the shipped artifacts, whose SHA-256s are listed in
  `notebooks/data/monti_overlap/ARTIFACTS.json`.

Match on `CHEMICAL_NAME`, never on `refmet_name`. Monti's Table S03, row 693, labels
(n(1)+n(8))-acetylspermidine as "Asparagine" in the RefMet name column; a join on that column
silently manufactures a wrong link.

In [4]:
PUBLIC_INPUTS = {
    "moesm5": (
        "https://static-content.springer.com/esm/art%3A10.1007%2Fs11357-026-02174-2/"
        "MediaObjects/11357_2026_2174_MOESM5_ESM.xlsx",
        "365b7cf304529e7cfa1619134d9d11cc9dade28386e9ea3e8b2dc6bc71eb8457",
    ),
    "moesm6": (
        "https://static-content.springer.com/esm/art%3A10.1007%2Fs11357-026-02174-2/"
        "MediaObjects/11357_2026_2174_MOESM6_ESM.xlsx",
        "81b4e3348b650681adc107b23866e2acfbb6ddf8654376b3dba035612dbc3045",
    ),
    "watanabe2023_supp_data2": (
        "https://static-content.springer.com/esm/art%3A10.1038%2Fs41591-023-02248-0/"
        "MediaObjects/41591_2023_2248_MOESM3_ESM.xlsx",
        "604dfca2a4601916f3db9964616063a1b471b9b68c2d221b73eb32fab89136f0",
    ),
}
SPREADSHEET_SHA256 = "83a717846fdb0a4900808e5aed1e7c9edbc23ad8e20b1bc857c4f91763e36851"


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


INPUT_DIR.mkdir(parents=True, exist_ok=True)
paths = {}
for key, (url, expected) in PUBLIC_INPUTS.items():
    path = INPUT_DIR / f"{key}.xlsx"
    if not path.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=120) as response:
            path.write_bytes(response.read())
    got = sha256(path)
    if got != expected:
        raise RuntimeError(f"{key}: sha256 {got} does not match the pinned {expected}")
    paths[key] = path
    print(f"{key:26s} sha256 OK  {path}")

if COHORT_SPREADSHEET:
    got = sha256(Path(COHORT_SPREADSHEET))
    if got != SPREADSHEET_SHA256:
        raise RuntimeError(f"4-cohort spreadsheet sha256 {got} != pinned {SPREADSHEET_SHA256}")
    print(f"{'4-cohort spreadsheet':26s} sha256 OK  {COHORT_SPREADSHEET}")
else:
    print(f"{'4-cohort spreadsheet':26s} not supplied: Xu, LLFS, BLSA are replay-only")

artifacts = json.loads((SHIPPED / "ARTIFACTS.json").read_text())
for name, expected in artifacts["sha256"].items():
    got = sha256(SHIPPED / name)
    if got != expected:
        raise RuntimeError(f"shipped artifact {name} has changed: {got} != {expected}")
print(f"\nshipped artifacts: {len(artifacts['sha256'])} files, sha256 OK")

moesm5                     sha256 OK  /home/trentleslie/monti_inputs/moesm5.xlsx
moesm6                     sha256 OK  /home/trentleslie/monti_inputs/moesm6.xlsx


watanabe2023_supp_data2    sha256 OK  /home/trentleslie/monti_inputs/watanabe2023_supp_data2.xlsx
4-cohort spreadsheet       not supplied: Xu, LLFS, BLSA are replay-only

shipped artifacts: 6 files, sha256 OK


---

## Part 3: Layer (a), overlap. Linked-count coverage, not correctness

Three columns per cohort:

- **Monti published**: the `# Overlap` column of MOESM6 `Table 2. Datasets`, NECS as the
  reference row. The next cell reads it straight from the downloaded supplement. The Methods prose
  gives different figures for two pairs (Xu 385, BLSA 188); the table, Table S03 and the cohort
  descriptions agree with each other and are used instead.
- **Arm M (BioMapper)**: NECS names that link to at least one cohort name, where a link means the
  two names resolved to identifier sets that intersect. Structure-encoding namespaces (InChIKey,
  InChI, SMILES) are excluded from linking, so the certificate in layer (b) is not reading back the
  quantity that formed the link.
- **Arm B (re-derived)**: Monti's own stated method, reconstructed on the same rows. It is a
  controlled variable we compute, not a published number.

A higher Arm M count means more links, not more correct links. Layer (b) is the correctness check.

In [5]:
manifest = json.loads((SHIPPED / "manifest.json").read_text())
results = manifest["results"]

# Monti's published comparator, read live from the downloaded MOESM6.
table2 = pd.read_excel(paths["moesm6"], sheet_name="Table 2. Datasets")
table2["cohort"] = table2["Dataset (Platform)"].str.extract(r"^(\w+)", expand=False).str.lower()
published_from_moesm6 = (
    table2.replace({"cohort": {"xu": "xuetal"}})
    .set_index("cohort")["# Overlap"]
    .dropna()
    .astype(int)
)

rows = []
for cohort in ("arivale", "xuetal", "llfs", "blsa"):
    r = results[cohort]
    if int(published_from_moesm6[cohort]) != r["monti_published"]:
        raise RuntimeError(f"{cohort}: MOESM6 says {published_from_moesm6[cohort]}, pinned run "
                           f"used {r['monti_published']}")
    rows.append({
        "cohort": cohort,
        "panel loaded / published": f"{r['panel_n_loaded']} / {r['panel_n_published']}",
        "Monti published": r["monti_published"],
        "Arm M NECS-linked": r["arm_m_necs_linked"],
        "Arm M - published": r["arm_m_vs_published"],
        "Arm B re-derived": r["arm_b_rederived"],
        "Arm B method": r["arm_b_method"],
        "comparator label": r["comparator_independence"]["label"],
    })
overlap = pd.DataFrame(rows)
print(overlap.to_string(index=False))
print("\nMonti published values re-read from MOESM6: match the pinned run.")

 cohort panel loaded / published  Monti published  Arm M NECS-linked  Arm M - published  Arm B re-derived                      Arm B method   comparator label
arivale                659 / 626              615                616                  1               583  CHEMICAL_NAME (case-insensitive) accuracy_candidate
 xuetal                821 / 821              432                775                343               470    CHEMICAL_NAME (case-sensitive) accuracy_candidate
   llfs                364 / 408              163                134                -29               144 RefMet join (drop_na before join)           coverage
   blsa                468 / 468               99                101                  2                79 RefMet join (drop_na before join)           coverage

Monti published values re-read from MOESM6: match the pinned run.


Reading the table:

- **Arivale** is the pair where the comparator is Metabolon `CHEMICAL_NAME` curation, a vocabulary
  KRAKEN does not ingest, so it is labelled `accuracy_candidate`: a like-for-like count, still not a
  correctness claim on its own.
- **LLFS and BLSA** use a RefMet join as the comparator, and RefMet is ingested into the graph being
  measured and is the resolver's source-weighting target. Those rows are **coverage**, full stop.
- **LLFS**: the paper profiled 408 metabolites; the spreadsheet ships the 364 RefMet-standardizable
  subset, so the loaded panel is 364. There is no "345" anywhere in the paper.
- **Xu**: Arm M links far more NECS names than Monti reported. Because Xu is names-only, none of
  those links can be structurally checked here, so the excess is unverified coverage, not evidence
  of better matching.

The next cell recomputes Arm B for Arivale from the two public files alone, with no API call. It
should equal the replayed value.

In [6]:
from biomapper.benchmarks.adapters.cohort_panel import (
    ARIVALE,
    SPREADSHEET_COHORTS,
    load_cohort_panel,
)
from biomapper.benchmarks.scorers.arm_b_baseline import arm_b_overlap

moesm5 = pd.read_excel(paths["moesm5"], dtype=str).fillna("")
necs_panel = load_cohort_panel(
    pd.DataFrame({"necs": moesm5["CHEMICAL_NAME"]}), SPREADSHEET_COHORTS["necs"]
)
arivale_frame = pd.read_excel(
    paths["watanabe2023_supp_data2"], sheet_name="Arivale_Metabolomics", dtype=str
).fillna("")
arivale_panel = load_cohort_panel(arivale_frame, ARIVALE)

arm_b = arm_b_overlap("arivale", necs_panel.names, arivale_panel.names)
print(f"NECS panel from MOESM5       {len(necs_panel.names)} named metabolites "
      f"(pinned run: {manifest['panel_sizes_loaded']['necs']})")
print(f"Arivale panel from Watanabe  {len(arivale_panel.names)} "
      f"(pinned run: {manifest['panel_sizes_loaded']['arivale']})")
print(f"Arm B, Arivale, recomputed   {arm_b.count}  [{arm_b.method}]")
print(f"Arm B, Arivale, pinned       {results['arivale']['arm_b_rederived']}")
assert arm_b.count == results["arivale"]["arm_b_rederived"]

NECS panel from MOESM5       1213 named metabolites (pinned run: 1213)
Arivale panel from Watanabe  659 (pinned run: 659)
Arm B, Arivale, recomputed   583  [CHEMICAL_NAME (case-insensitive)]
Arm B, Arivale, pinned       583


---

## Part 4: Layer (b), certification. Arivale only

For each NECS to Arivale link, an InChIKey is obtained for each side **from outside the graph**:
the NECS side from the curated key in MOESM5, the Arivale side by looking up its vendor PubChem CID
or HMDB accession in PubChem. The link is then

- **certified** when the two InChIKey first blocks agree,
- **refuted** when both exist and disagree,
- **refused** when either side has no independent structure. Refusal is reported, never guessed.

The headline is the **certified rate over adjudicable links** (certified plus refuted), quoted
together with **coverage** (adjudicable over all links). It is not a rate over all links.

The replayed report was produced by `cross_cohort_certify` at the pinned commit, which includes the
fix for legacy 5-digit HMDB accessions (biomapper PR #17). A report computed before that fix is
shipped too, for the methods note only.

In [7]:
def arivale_split(report: dict) -> dict:
    a = report["cohorts"]["arivale"]
    card = a["independent_source_card"]
    if card["transient_failures"] != 0:
        raise RuntimeError(
            f"{card['transient_failures']} PubChem lookups failed transiently; this split is a run "
            "artifact and must not be quoted. Re-run certification."
        )
    return {
        "links": a["n_links"],
        "adjudicable": a["adjudicable"],
        "coverage": a["adjudicable"] / a["n_links"],
        "certified": a["certified"],
        "refuted": a["refuted"],
        "refused": a["refused"],
        "certified_over_adjudicable": a["certified_rate_over_adjudicable"],
        "cohort_blocks_resolved": f"{card['blocks_resolved']} / {card['names_requested']}",
        "transient_failures": card["transient_failures"],
    }


report = json.loads((SHIPPED / "certificate_report.json").read_text())
prefix = json.loads((SHIPPED / "certificate_report.pre_hmdb_fix.json").read_text())
split = pd.DataFrame(
    {"reported": arivale_split(report), "pre-fix (methods note)": arivale_split(prefix)}
)
print(split.to_string())

a = report["cohorts"]["arivale"]
print(f"\nNECS<->Arivale: {a['certified']} certified / {a['refuted']} refuted / "
      f"{a['refused']} refused; "
      f"coverage {a['adjudicable']}/{a['n_links']} = {a['adjudicable'] / a['n_links']:.1%}; "
      f"certified over adjudicable {a['certified_rate_over_adjudicable']:.1%}")
print(f"stereo checked: {a['stereo_checked_any']}  |  {a['comparison_granularity']}")

print("\nRefusal classes:")
for k, v in sorted(a["refusal_classes"].items(), key=lambda kv: -kv[1]):
    print(f"  {k:48s} {v}")

print("\nCohorts refused by construction (names-only panels):")
for cohort in ("xuetal", "llfs", "blsa"):
    c = report["cohorts"][cohort]
    print(f"  {cohort:8s} {c['n_links']} links, certifiable={c['certifiable']}")

                             reported pre-fix (methods note)
links                            1792                   1792
adjudicable                       736                    536
coverage                     0.410714               0.299107
certified                         452                    398
refuted                           284                    138
refused                          1056                   1256
certified_over_adjudicable    0.61413               0.742537
cohort_blocks_resolved      468 / 591              405 / 591
transient_failures                  0                      0

NECS<->Arivale: 452 certified / 284 refuted / 1056 refused; coverage 736/1792 = 41.1%; certified over adjudicable 61.4%
stereo checked: False  |  InChIKey first block (connectivity) only. Not tautomer- or charge-invariant: over-flags tautomers as refuted, silently accepts stereoisomer errors.

Refusal classes:
  no_independent_structure_either_side             464
  cohort_panel_has_no_

### What the pre-fix column is, and why it is not the headline

The Arivale panel ships legacy 5-digit HMDB accessions (for example `HMDB13127`). PubChem resolves
only the 7-digit form (`HMDB0013127`), so before the fix those rows came back as clean misses and
their links were refused as if no structure existed. The rows affected were mostly lipids, and
lipid links are the ones most likely to refute. Excluding them raised the certified rate. With the
fix, the added adjudicable links are dominated by cross-species lipid links, for example a GPE
linked to a GPI or sphingomyelins with different chains.

### Re-adjudication of the non-certified cases

Each refused or refuted case is re-examined against a second outside source (PubChem's name index)
to say *why* it failed: a lipid chain annotation difference, a suspect NECS gold key, a conjugate
linked to its parent, and so on.

In [8]:
readj = json.loads((SHIPPED / "readjudication_arivale.json").read_text())
print(f"cases examined: {readj['cases_examined']}")
for outcome, n in sorted(readj["outcomes"].items(), key=lambda kv: -kv[1]):
    print(f"  {outcome:44s} {n}")
print(f"\ncaveat: {readj['caveat']}")

cases examined: 1340
  not_adjudicable_by_construction              838
  differing_lipid_chain_annotation             329
  distinct_vendor_names_structure_unverified   130
  outside_source_unresolved                    32
  necs_gold_suspect                            9
  both_sides_disagree_with_outside             1
  conjugate_linked_to_parent                   1

caveat: a first-block comparison over-flags tautomers and charge variants as refuted, and silently accepts a stereoisomer error when a side has no stereo layer. The second failure mode is not recoverable by this re-adjudication.


---

## Part 5: What the certificate is not

- **Not fully knowledge-graph independent.** This step never reads the linking node's own
  InChIKey (the API's per-entity `ResolutionCertificate` does, and is a different instrument). It
  is still not fully independent of the graph, for two reasons. The graph decides which pairs are
  linked and therefore which pairs get certified. And the preferred NECS gold column, MOESM5
  `inchi_key`, sits among columns (`formula`, `exactmass`, `smiles`, `pubchem_cid`, `refmet`) that
  appear to be RefMet annotations; RefMet is ingested into KRAKEN, so gold and graph may share an
  upstream source. The Arivale side, looked up in PubChem from vendor ids, is outside the graph.
- **First-block comparison** (connectivity only) over-flags tautomers and charge variants as
  refuted, and silently accepts stereoisomer errors. No stereochemistry is checked.
- **The NECS gold carries roughly 5% InChIKey errors.** A refutation is not by itself evidence that
  BioMapper is wrong; re-adjudication flags `necs_gold_suspect` cases separately.
- **Third-party dependency.** Certification depends on PubChem PUG-REST, which has no SLA. A run
  with any transient lookup failure is refused by the check in Part 4 rather than reported.

---

## Part 6: Live subset check

This resolves 17 NECS and 50 Arivale names against the keyless deployment, links them, and
compares each name's identifier set with the pinned run. The names are fixed in
`subset_expected.json` and come only from the public files. They include
(n(1)+n(8))-acetylspermidine, whose pinned link to Arivale "asparagine" the certificate refutes,
and one NECS sphingomyelin that links to about thirty Arivale sphingomyelins through a class-level
node.

Outputs are written to `OUT_DIR` by default.

In [9]:
from biomapper.benchmarks.api_mapper import ApiMapper
from biomapper.benchmarks.cross_cohort import REQUEST
from biomapper.benchmarks.scorers.cross_cohort_overlap import link_by_intersection, row_curie_set

subset = json.loads((SHIPPED / "subset_expected.json").read_text())


def resolve(names: list[str], label: str) -> dict[str, frozenset]:
    mapper = ApiMapper(endpoint=API_ENDPOINT, api_key=None, batch_size=20)
    tsv, _stats = mapper.map_dataset_to_kg(
        dataset=pd.DataFrame({"name": names}),
        entity_type=REQUEST["entity_type"],
        name_column="name",
        provided_id_columns=[],
        vocab=REQUEST["vocab"],
        annotation_mode=REQUEST["annotation_mode"],
        output_dir=OUT_DIR,
        output_prefix=f"subset_{label}",
    )
    mapped = pd.read_csv(tsv, sep="\t", dtype=str).fillna("")
    assert mapped["name"].tolist() == names, "batch order drifted"
    return {r["name"]: row_curie_set(r) for _, r in mapped.iterrows()}


if RUN_SUBSET:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    live = {side: resolve(list(subset[side]), side) for side in ("necs", "arivale")}
    changed = [
        (side, name)
        for side in ("necs", "arivale")
        for name, pinned in subset[side].items()
        if set(pinned) != set(live[side][name])
    ]
    overlap_live = link_by_intersection(live["necs"], live["arivale"])
    got = {"links": overlap_live.n_links, "necs_linked": overlap_live.n_a_linked,
           "arivale_linked": overlap_live.n_b_linked}
    print(f"names whose identifier set differs from the pinned run: {len(changed)}")
    for side, name in changed:
        print(f"  {side}: {name}")
    print(f"subset overlap live   {got}")
    print(f"subset overlap pinned {subset['expected']}")
    (OUT_DIR / "subset_result.json").write_text(json.dumps({
        "live_build": live_build, "live_build_matches_pinned": LIVE_BUILD_MATCHES,
        "changed": changed, "overlap_live": got, "overlap_pinned": subset["expected"],
    }, indent=2))
    print(f"\nsaved: {OUT_DIR / 'subset_result.json'}")
    if LIVE_BUILD_MATCHES and (changed or got != subset["expected"]):
        print("WARNING: same build, different answers. Resolution should be deterministic for a "
              "fixed graph state; investigate before quoting anything.")

names whose identifier set differs from the pinned run: 0
subset overlap live   {'links': 77, 'necs_linked': 13, 'arivale_linked': 47}
subset overlap pinned {'links': 77, 'necs_linked': 13, 'arivale_linked': 47}

saved: /home/trentleslie/monti_overlap_runs/20260928T200138Z/subset_result.json


Now certify the subset's links with the same functions the full run uses: the NECS gold from
MOESM5, the Arivale side from its vendor ids through PubChem.

In [10]:
from biomapper.benchmarks.cross_cohort_certify import (
    adjudicate_cases,
    arivale_independent_blocks,
    necs_gold_blocks,
)

if RUN_SUBSET:
    necs_blocks, _necs_card = necs_gold_blocks(paths["moesm5"])
    cohort_blocks, cohort_card = arivale_independent_blocks(
        paths["watanabe2023_supp_data2"], {link.b_name for link in overlap_live.links}
    )
    if cohort_card["transient_failures"]:
        print(f"{cohort_card['transient_failures']} transient PubChem failures: the subset split "
              "below is not meaningful. Re-run this cell.")
    cases = adjudicate_cases(list(overlap_live.links), necs_blocks, cohort_blocks)
    cases.to_csv(OUT_DIR / "subset_certificate_cases.csv", index=False)
    print(cases["verdict"].value_counts().to_string())
    print()
    print(cases.loc[cases["necs_name"].str.contains("acetylspermidine"),
                    ["necs_name", "cohort_name", "verdict", "necs_block", "cohort_block"]]
          .to_string(index=False))
    print(f"\nsaved: {OUT_DIR / 'subset_certificate_cases.csv'}")

verdict
refused      64
certified    10
refuted       3

                     necs_name                    cohort_name verdict     necs_block   cohort_block
(n(1) + n(8))-acetylspermidine (N(1) + N(8))-acetylspermidine refuted FONIWJIDLJEJTL MQTAVJHICJWXBR
(n(1) + n(8))-acetylspermidine                     asparagine refuted FONIWJIDLJEJTL DCXYFEDJOCDNAF

saved: /home/trentleslie/monti_overlap_runs/20260928T200138Z/subset_certificate_cases.csv


---

## Part 7: Full run (off by default)

Set `RUN_FULL = True` and `MONTI_COHORT_SPREADSHEET` to reproduce every number above from
scratch. Resolution is checkpointed per panel, so an interrupted run resumes. Budget about 5.6
hours of API time for resolution, then several minutes each for certification and
re-adjudication. Everything is written under `OUT_DIR / "full"`.

The NECS column of the spreadsheet is identical to MOESM5 `CHEMICAL_NAME`; the full run uses the
spreadsheet because that is what the pinned run used.

In [11]:
import subprocess

if RUN_FULL:
    if not COHORT_SPREADSHEET:
        raise RuntimeError("RUN_FULL needs MONTI_COHORT_SPREADSHEET (the 4-cohort names file)")
    full = OUT_DIR / "full"
    full.mkdir(parents=True, exist_ok=True)
    common = ["--spreadsheet", COHORT_SPREADSHEET,
              "--arivale-xlsx", str(paths["watanabe2023_supp_data2"])]
    steps = [
        *[["-m", "biomapper.benchmarks.cross_cohort", "--out-dir", str(full), "--panel", p, *common]
          for p in ("necs", "arivale", "xuetal", "llfs", "blsa")],
        ["-m", "biomapper.benchmarks.cross_cohort", "--out-dir", str(full), "--link-only", *common],
        ["-m", "biomapper.benchmarks.cross_cohort_certify", "--run-dir", str(full),
         "--moesm5", str(paths["moesm5"]), "--arivale-xlsx", str(paths["watanabe2023_supp_data2"])],
        ["-m", "biomapper.benchmarks.cross_cohort_readjudicate", "--run-dir", str(full),
         "--cohort", "arivale"],
    ]
    for step in steps:
        print("$ python", " ".join(step), flush=True)
        subprocess.run([sys.executable, *step], check=True)
    print(f"\nfull run saved under {full}")
else:
    print("RUN_FULL is False: the replay above is the full result.")

RUN_FULL is False: the replay above is the full result.


---

## Guardrails for quoting these numbers

- Quote the **certified rate over adjudicable links together with coverage** (adjudicable over all
  links). It is a correctness check on the adjudicable subset, not accuracy over all links.
- Label the certificate **not fully knowledge-graph independent** (Part 5): the graph selects the
  pairs, and the NECS gold may share RefMet as an upstream source with the graph.
- InChIKey **first-block** comparison over-flags tautomers and silently accepts stereoisomer errors.
- The NECS gold carries about **5% InChIKey errors**.
- **LLFS**: 408 published, 364 loaded. Never "345".
- Match on **`CHEMICAL_NAME`**, not `refmet_name` (Table S03, row 693).
- The overlap layer is **linked-count coverage**. Xu, LLFS and BLSA links are never structurally
  certified here.
- Certification is recomputed per link on this build; it is **not inherited** from an earlier
  build's run, whose link set differs.